In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from shapely.geometry import Polygon, MultiPolygon
from skimage.draw import polygon as draw_polygon
from skimage.measure import regionprops


# ============================================================
# Helper functions
# ============================================================

def clean_contour(contour):
    """
    Convert contour coordinates to a clean NumPy array.
    """

    points = np.asarray(contour, dtype=np.float64)

    if points.ndim != 2 or points.shape[1] != 2:
        return None

    # Remove consecutive duplicate points.
    keep = np.ones(len(points), dtype=bool)
    keep[1:] = np.any(points[1:] != points[:-1], axis=1)
    points = points[keep]

    if len(points) < 3:
        return None

    return points


def contour_to_local_mask(points, padding=2):
    """
    Convert one contour into a small binary mask.

    The mask is needed to calculate eccentricity.
    """

    x = points[:, 0]
    y = points[:, 1]

    x_min = int(np.floor(x.min())) - padding
    y_min = int(np.floor(y.min())) - padding

    x_max = int(np.ceil(x.max())) + padding
    y_max = int(np.ceil(y.max())) + padding

    width = x_max - x_min + 1
    height = y_max - y_min + 1

    if width <= 0 or height <= 0:
        return None

    mask = np.zeros((height, width), dtype=np.uint8)

    # Shift whole-slide coordinates into local mask coordinates.
    local_x = x - x_min
    local_y = y - y_min

    rr, cc = draw_polygon(
        local_y,
        local_x,
        shape=mask.shape
    )

    mask[rr, cc] = 1

    return mask


def calculate_morphology(contour):
    """
    Calculate area, perimeter, circularity,
    solidity, eccentricity, and contour centroid.
    """

    points = clean_contour(contour)

    if points is None:
        return None

    poly = Polygon(points)

    # Attempt to repair invalid polygons.
    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        return None

    # Keep the largest component if repair creates a MultiPolygon.
    if isinstance(poly, MultiPolygon):
        poly = max(poly.geoms, key=lambda p: p.area)

    if not isinstance(poly, Polygon):
        return None

    area = float(poly.area)
    perimeter = float(poly.length)

    if area <= 0 or perimeter <= 0:
        return None

    circularity = 4.0 * np.pi * area / (perimeter ** 2)

    hull_area = float(poly.convex_hull.area)

    solidity = (
        area / hull_area
        if hull_area > 0
        else np.nan
    )

    # Eccentricity requires a binary mask.
    eccentricity = np.nan

    mask = contour_to_local_mask(points)

    if mask is not None and mask.any():
        regions = regionprops(mask)

        if regions:
            eccentricity = float(regions[0].eccentricity)

    return {
        "contour_centroid_x": float(poly.centroid.x),
        "contour_centroid_y": float(poly.centroid.y),
        "area_px2": area,
        "perimeter_px": perimeter,
        "circularity": circularity,
        "solidity": solidity,
        "eccentricity": eccentricity
    }


# ============================================================
# Main pipeline
# ============================================================

def build_cellvit_dataframe(
    cell_detection_path,
    cells_geojson_path
):
    """
    Build the final CellViT dataframe with coordinates,
    phenotype, and five morphology features.
    """

    cell_detection_path = Path(cell_detection_path)
    cells_geojson_path = Path(cells_geojson_path)

    # --------------------------------------------------------
    # 1. Load cell_detection.json
    # --------------------------------------------------------

    with cell_detection_path.open("r") as file:
        detection_data = json.load(file)

    type_map = {
        int(type_id): cell_type
        for type_id, cell_type
        in detection_data["type_map"].items()
    }

    detection_rows = []

    for cell_number, cell in enumerate(detection_data["cells"]):

        centroid_x, centroid_y = cell["centroid"]
        cell_type_id = int(cell["type"])

        detection_rows.append({
            "cell_id": f"cell_{cell_number:06d}",
            "x": float(centroid_x),
            "y": float(centroid_y),
            "cell_type_id": cell_type_id,
            "cell_type": type_map.get(
                cell_type_id,
                "Unknown"
            )
        })

    obs_df = pd.DataFrame(detection_rows)


    # --------------------------------------------------------
    # 2. Load cells.geojson and extract morphology
    # --------------------------------------------------------

    with cells_geojson_path.open("r") as file:
        geojson_features = json.load(file)

    morphology_rows = []

    # Each top-level feature represents one phenotype.
    for feature in geojson_features:

        cell_type = (
            feature["properties"]
            ["classification"]
            ["name"]
        )

        geometry = feature["geometry"]

        if geometry["type"] != "MultiPolygon":
            continue

        # Each polygon corresponds to one cell.
        for polygon in geometry["coordinates"]:

            if not polygon:
                continue

            # The first ring is the exterior contour.
            contour = polygon[0]

            morphology = calculate_morphology(contour)

            if morphology is None:
                continue

            morphology_rows.append({
                "cell_type": cell_type,
                **morphology
            })

    morphology_df = pd.DataFrame(morphology_rows)


    # --------------------------------------------------------
    # 3. Match contours to detections by phenotype and centroid
    # --------------------------------------------------------

    morphology_df["cell_id"] = None
    morphology_df["matching_distance_px"] = np.nan

    for cell_type in morphology_df["cell_type"].unique():

        detections = obs_df[
            obs_df["cell_type"] == cell_type
        ].copy()

        contours = morphology_df[
            morphology_df["cell_type"] == cell_type
        ].copy()

        if detections.empty or contours.empty:
            continue

        tree = cKDTree(
            detections[["x", "y"]].to_numpy()
        )

        distances, positions = tree.query(
            contours[
                [
                    "contour_centroid_x",
                    "contour_centroid_y"
                ]
            ].to_numpy(),
            k=1
        )

        matched_ids = (
            detections.iloc[positions]["cell_id"]
            .to_numpy()
        )

        morphology_df.loc[
            contours.index,
            "cell_id"
        ] = matched_ids

        morphology_df.loc[
            contours.index,
            "matching_distance_px"
        ] = distances


    # --------------------------------------------------------
    # 4. Merge morphology into the detection dataframe
    # --------------------------------------------------------

    morphology_for_merge = morphology_df[
        [
            "cell_id",
            "area_px2",
            "perimeter_px",
            "circularity",
            "solidity",
            "eccentricity"
        ]
    ]

    final_df = obs_df.merge(
        morphology_for_merge,
        on="cell_id",
        how="left",
        validate="one_to_one"
    )


    # Arrange the final columns.
    final_df = final_df[
        [
            "cell_id",
            "x",
            "y",
            "cell_type_id",
            "cell_type",
            "area_px2",
            "perimeter_px",
            "circularity",
            "solidity",
            "eccentricity"
        ]
    ]


    # --------------------------------------------------------
    # 5. Basic validation
    # --------------------------------------------------------

    print("Detected cells:", len(obs_df))
    print("Extracted contours:", len(morphology_df))
    print("Final dataframe shape:", final_df.shape)

    print(
        "Unmatched contours:",
        morphology_df["cell_id"].isna().sum()
    )

    print(
        "Duplicate matches:",
        morphology_df["cell_id"]
        .dropna()
        .duplicated()
        .sum()
    )

    print(
        "Missing morphology rows:",
        final_df["area_px2"].isna().sum()
    )

    return final_df

In [ ]:
cell_detection_path = (
    "../../../1_cellvit/output/cellvit_out/SPA51/cell_detection.json"
)

cells_geojson_path = (
    "../../../1_cellvit/output/cellvit_out/SPA51/cells.geojson"
)


final_obs_df = build_cellvit_dataframe(
    cell_detection_path,
    cells_geojson_path
)